# 宏观变量与 GDP raw data 处理

本 notebook 汇总宏观变量部分的全部处理流程，最终只输出一个文件：`宏观变量&GDP raw data.csv`。


处理内容：读取两个 Excel，按日期合并，处理固定资产投资累计值，生成 GDP 同比列，映射到 `1980-01` 至 `2026-04` 的月度数据库，并删除中间变量。


## 1. 路径与参数


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "AGENTS.md").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "AGENTS.md").exists():
            PROJECT_ROOT = parent
            break

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
DATA_DIR = next(path for path in RAW_ROOT.iterdir() if path.is_dir() and any(path.glob("GDP&*.xlsx")))
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GDP_MACRO_FILE = next(DATA_DIR.glob("GDP&*.xlsx"))
RETAIL_FAI_FILE = next(path for path in DATA_DIR.glob("*.xlsx") if path.name != GDP_MACRO_FILE.name and "&" in path.name)
FINAL_OUTPUT_FILE = OUTPUT_DIR / ("".join(map(chr, [23439, 35266, 21464, 37327])) + "&GDP raw data.csv")

START_MONTH = "1980-01"
END_MONTH = "2026-04"

print(GDP_MACRO_FILE)
print(RETAIL_FAI_FILE)
print(FINAL_OUTPUT_FILE)


E:\财遇见你\大四\申万实习\动态因子模型\动态因子模型\data\raw\1980年至今数据\GDP&宏观基本面.xlsx
E:\财遇见你\大四\申万实习\动态因子模型\动态因子模型\data\raw\1980年至今数据\宏观基本面-社会消费品零售总额&固定资产投资.xlsx
E:\财遇见你\大四\申万实习\动态因子模型\动态因子模型\data\processed\processed\宏观变量&GDP raw data.csv


## 2. 生成月度日期序列


In [2]:
months = pd.period_range(START_MONTH, END_MONTH, freq="M")
month_sequence = pd.DataFrame({
    "month": months.astype(str),
    "month_end": months.to_timestamp("M"),
})

print(month_sequence.shape)
display(month_sequence.head())
display(month_sequence.tail())


(556, 2)


,month,month_end
0,1980-01,1980-01-31
1,1980-02,1980-02-29
2,1980-03,1980-03-31
3,1980-04,1980-04-30
4,1980-05,1980-05-31


,month,month_end
551,2025-12,2025-12-31
552,2026-01,2026-01-31
553,2026-02,2026-02-28
554,2026-03,2026-03-31
555,2026-04,2026-04-30


## 3. 读取两个 Excel

`GDP&宏观基本面.xlsx` 为 Wind 常见格式：第 1 行变量名，第 2 行指标代码，第 3 行开始是日期和数据。

`宏观基本面-社会消费品零售总额&固定资产投资.xlsx` 为补充表：第 1 行变量名，第 2 行频率，第 3 行单位，第 4 行指标 ID，第 5 行来源，第 6 行开始是日期和数据。


In [3]:
def read_gdp_macro_file(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="Sheet2", header=None)
    variable_names = raw.iloc[0].tolist()
    data = raw.iloc[2:].copy()
    data.columns = variable_names
    date_col = variable_names[0]
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
    data = data.dropna(subset=[date_col])
    data[date_col] = data[date_col].dt.to_period("M").dt.to_timestamp("M")
    data = data.sort_values(date_col).drop_duplicates(subset=[date_col], keep="last")
    return data.rename(columns={date_col: "month_end"})


def read_retail_fai_file(path: Path) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name="Sheet1", header=None)
    variable_names = raw.iloc[0].tolist()
    data = raw.iloc[5:].copy()
    data.columns = variable_names
    date_col = variable_names[0]
    data[date_col] = pd.to_datetime(data[date_col], errors="coerce")
    data = data.dropna(subset=[date_col])
    data[date_col] = data[date_col].dt.to_period("M").dt.to_timestamp("M")
    data = data.sort_values(date_col).drop_duplicates(subset=[date_col], keep="last")
    for col in variable_names[1:]:
        data[col] = pd.to_numeric(data[col], errors="coerce")
    return data.rename(columns={date_col: "month_end"})


gdp_macro_raw = read_gdp_macro_file(GDP_MACRO_FILE)
retail_fai_raw = read_retail_fai_file(RETAIL_FAI_FILE)

print(gdp_macro_raw.shape)
print(retail_fai_raw.shape)
display(gdp_macro_raw.head())
display(retail_fai_raw.head())


(473, 13)
(394, 3)


,month_end,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:累计同比,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率
2,1987-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.1,NaN,NaN
3,1987-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,NaN,NaN
4,1987-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8,NaN,NaN
5,1987-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.7,NaN,NaN
6,1987-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.6,NaN,NaN


,month_end,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:累计值
398,1992-02-29,0.0,125.82
397,1992-03-31,0.0,315.36
396,1992-04-30,0.0,523.81
395,1992-05-31,0.0,799.11
394,1992-06-30,0.0,1170.98


## 4. 按日期合并并删除重复社零列

两个文件中都有 `中国:社会消费品零售总额:当月同比(1-2月合并)`，且完全一致。合并后删除 `__gdp_file` 重复列，只保留补充表原名列。


In [4]:
merged_raw = pd.merge(
    gdp_macro_raw,
    retail_fai_raw,
    on="month_end",
    how="outer",
    suffixes=("__gdp_file", ""),
)

duplicate_retail_col = "中国:社会消费品零售总额:当月同比(1-2月合并)__gdp_file"
if duplicate_retail_col in merged_raw.columns:
    merged_raw = merged_raw.drop(columns=[duplicate_retail_col])

merged_raw = merged_raw.sort_values("month_end").reset_index(drop=True)

print(merged_raw.shape)
display(merged_raw.head())
display(merged_raw.tail())


(473, 14)


,month_end,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:固定资产投资完成额:累计同比,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:累计值
0,1987-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.1,NaN,NaN,NaN,NaN
1,1987-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.4,NaN,NaN,NaN,NaN
2,1987-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.8,NaN,NaN,NaN,NaN
3,1987-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.7,NaN,NaN,NaN,NaN
4,1987-05-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.6,NaN,NaN,NaN,NaN


,month_end,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:固定资产投资完成额:累计同比,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:累计值
468,2026-01-31,NaN,NaN,NaN,49.3,49.5,6.3,NaN,NaN,0.2,-1.4,5.2,NaN,NaN
469,2026-02-28,NaN,NaN,NaN,49,49.7,6.3,NaN,1.8,1.3,-0.9,5.3,2.8,52721.0
470,2026-03-31,334192.9,335748.7,4.89,50.4,50.2,5.7,3.515013,1.7,1,0.5,5.4,1.7,102708.0
471,2026-04-30,NaN,NaN,NaN,50.3,49.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
472,2026-06-30,NaN,NaN,4.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. 固定资产投资累计值拆分并计算当月同比

规则：1 月为空；2 月累计值视为 1-2 月合计值，同比写在 2 月；3 月及以后用 `本月累计值 - 上月累计值` 拆分当月值，再计算当月同比。


In [5]:
FAI_CUM_COL = "中国:固定资产投资完成额:累计值"
FAI_MONTH_VALUE_COL = "中国:固定资产投资完成额:当月值_由累计值拆分"
FAI_MONTH_YOY_COL = "中国:固定资产投资完成额:当月同比_由累计值计算"

fai = retail_fai_raw[["month_end", FAI_CUM_COL]].copy()
fai["year"] = fai["month_end"].dt.year
fai["month_num"] = fai["month_end"].dt.month
fai = fai.sort_values("month_end")
fai["prev_cum_same_year"] = fai.groupby("year")[FAI_CUM_COL].shift(1)

fai[FAI_MONTH_VALUE_COL] = np.where(
    fai["month_num"] == 2,
    fai[FAI_CUM_COL],
    np.where(
        fai["month_num"] >= 3,
        fai[FAI_CUM_COL] - fai["prev_cum_same_year"],
        np.nan,
    ),
)

fai["month_period"] = fai["month_end"].dt.to_period("M")
fai = fai.set_index("month_period")
fai["last_year_month_value"] = fai[FAI_MONTH_VALUE_COL].shift(12)
fai[FAI_MONTH_YOY_COL] = fai[FAI_MONTH_VALUE_COL] / fai["last_year_month_value"] - 1
fai = fai.reset_index(drop=False)
fai["month_end"] = fai["month_period"].dt.to_timestamp("M")

fai_constructed = fai[["month_end", FAI_MONTH_VALUE_COL, FAI_MONTH_YOY_COL]].copy()
display(fai_constructed.tail(15))


,month_end,中国:固定资产投资完成额:当月值_由累计值拆分,中国:固定资产投资完成额:当月同比_由累计值计算
379,2024-11-30,42617.0,-0.039595
380,2024-12-31,48535.0,0.172201
381,2025-02-28,52619.0,0.246246
382,2025-03-31,50555.0,-0.005743
383,2025-04-30,43850.0,-0.108649
384,2025-05-31,44923.0,0.036071
385,2025-06-30,56707.0,0.271315
386,2025-07-31,39575.0,-0.310360
387,2025-08-31,37882.0,-0.102748
388,2025-09-30,45424.0,0.087375


## 6. 映射到 1980-01 至 2026-04 月度面板

季度变量只保留季度末月份，月度变量按月末日期对应，不使用目标结束月之后的数据。


In [6]:
def is_quarterly_series(dates: pd.Series, values: pd.Series) -> bool:
    valid_dates = dates[values.notna()]
    if valid_dates.empty:
        return False
    return set(valid_dates.dt.month.astype(int).tolist()).issubset({3, 6, 9, 12})


def map_series_to_monthly(dates: pd.Series, values: pd.Series, target_month_end: pd.DatetimeIndex, frequency: str) -> pd.Series:
    source = pd.DataFrame({"date": dates, "value": values}).dropna(subset=["date"])
    source = source.drop_duplicates(subset=["date"], keep="last")
    source = source[source["date"] <= target_month_end.max()]
    if frequency == "quarterly":
        source = source[source["date"].dt.month.isin([3, 6, 9, 12])]
    source = source.dropna(subset=["value"]).set_index("date")["value"]
    out = pd.Series(index=target_month_end, dtype="float64")
    common_idx = source.index.intersection(target_month_end)
    out.loc[common_idx] = source.loc[common_idx]
    return out


source = pd.merge(merged_raw, fai_constructed, on="month_end", how="left")

final_variables = [
    "中国:GDP:现价:当季值",
    "中国:GDP:不变价:当季值",
    "万得一致预测:中国:GDP:不变价:当季同比",
    "中国:制造业PMI",
    "中国:非制造业PMI:服务业",
    "中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分)",
    "中国:全社会用电量:当月同比",
    "中国:社会消费品零售总额:当月同比(1-2月合并)",
    "中国:固定资产投资完成额:累计同比",
    FAI_CUM_COL,
    FAI_MONTH_VALUE_COL,
    FAI_MONTH_YOY_COL,
    "中国:CPI:当月同比",
    "中国:PPI:当月同比",
    "中国:城镇调查失业率",
]
final_variables = [col for col in final_variables if col in source.columns]

target_month_end = pd.DatetimeIndex(month_sequence["month_end"])
mapped = month_sequence[["month"]].copy()

for col in final_variables:
    values = pd.to_numeric(source[col], errors="coerce")
    dates = source["month_end"]
    frequency = "quarterly" if is_quarterly_series(dates, values) else "monthly"
    mapped[col] = map_series_to_monthly(dates, values, target_month_end, frequency).to_numpy()

print(mapped.shape)
display(mapped.head())
display(mapped.tail())


(556, 16)


,month,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:累计同比,中国:固定资产投资完成额:累计值,中国:固定资产投资完成额:当月值_由累计值拆分,中国:固定资产投资完成额:当月同比_由累计值计算,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率
0,1980-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1980-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1980-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1980-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1980-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,month,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:累计同比,中国:固定资产投资完成额:累计值,中国:固定资产投资完成额:当月值_由累计值拆分,中国:固定资产投资完成额:当月同比_由累计值计算,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率
551,2025-12,387911.3,374444.5,4.48,50.1,49.7,5.2,2.770855,0.9,-3.8,485186.0,41151.0,-0.034399,0.8,-1.9,5.1
552,2026-01,NaN,NaN,NaN,49.3,49.5,6.3,NaN,NaN,NaN,NaN,NaN,NaN,0.2,-1.4,5.2
553,2026-02,NaN,NaN,NaN,49.0,49.7,6.3,NaN,2.8,1.8,52721.0,52721.0,0.086247,1.3,-0.9,5.3
554,2026-03,334192.9,335748.7,4.89,50.4,50.2,5.7,3.515013,1.7,1.7,102708.0,49987.0,-0.050020,1.0,0.5,5.4
555,2026-04,NaN,NaN,NaN,50.3,49.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. 生成 GDP 同比列

对 `中国:GDP:现价:当季值` 和 `中国:GDP:不变价:当季值` 生成同比列。季度 GDP 只在季度末月份有值，因此用 12 个月滞后值计算同比。


In [7]:
gdp_yoy_map = {
    "中国:GDP:现价:当季值": "中国:GDP:现价:当季值同比",
    "中国:GDP:不变价:当季值": "中国:GDP:不变价:当季值同比",
}

for value_col, yoy_col in gdp_yoy_map.items():
    if value_col in mapped.columns:
        mapped[yoy_col] = mapped[value_col] / mapped[value_col].shift(12) - 1

display(mapped[["month"] + [c for pair in gdp_yoy_map.items() for c in pair if c in mapped.columns]].tail(20))


,month,中国:GDP:现价:当季值,中国:GDP:现价:当季值同比,中国:GDP:不变价:当季值,中国:GDP:不变价:当季值同比
536,2024-09,341443.2,0.039589,325479.2,0.045587
537,2024-10,NaN,NaN,NaN,NaN
538,2024-11,NaN,NaN,NaN,NaN
539,2024-12,373512.4,0.045595,358465.4,0.053557
540,2025-01,NaN,NaN,NaN,NaN
541,2025-02,NaN,NaN,NaN,NaN
542,2025-03,318466.4,0.045780,306571.0,0.054262
543,2025-04,NaN,NaN,NaN,NaN
544,2025-05,NaN,NaN,NaN,NaN
545,2025-06,341395.3,0.038985,323458.2,0.052345


## 8. 保存最终 CSV

删除固定资产投资计算过程中的中间变量，只保留最终当月同比，并保存为 `宏观变量&GDP raw data.csv`。


In [8]:
df = mapped.copy()

drop_cols = [
    "中国:固定资产投资完成额:累计同比",
    "中国:固定资产投资完成额:累计值",
    "中国:固定资产投资完成额:当月值_由累计值拆分",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df = df.rename(columns={
    "中国:固定资产投资完成额:当月同比_由累计值计算": "中国:固定资产投资完成额:当月同比"
})

df.to_csv(FINAL_OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"saved final file: {FINAL_OUTPUT_FILE}")
print(f"shape: {df.shape}")
display(df.tail())


saved final file: E:\财遇见你\大四\申万实习\动态因子模型\动态因子模型\data\processed\processed\宏观变量&GDP raw data.csv
shape: (556, 15)


,month,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:当月同比,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率,中国:GDP:现价:当季值同比,中国:GDP:不变价:当季值同比
551,2025-12,387911.3,374444.5,4.48,50.1,49.7,5.2,2.770855,0.9,-0.034399,0.8,-1.9,5.1,0.038550,0.044576
552,2026-01,NaN,NaN,NaN,49.3,49.5,6.3,NaN,NaN,NaN,0.2,-1.4,5.2,NaN,NaN
553,2026-02,NaN,NaN,NaN,49.0,49.7,6.3,NaN,2.8,0.086247,1.3,-0.9,5.3,NaN,NaN
554,2026-03,334192.9,335748.7,4.89,50.4,50.2,5.7,3.515013,1.7,-0.050020,1.0,0.5,5.4,0.049382,0.095174
555,2026-04,NaN,NaN,NaN,50.3,49.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. 最终检查


In [9]:
final_df = pd.read_csv(FINAL_OUTPUT_FILE, encoding="utf-8-sig")

print("final file:", FINAL_OUTPUT_FILE)
print("shape:", final_df.shape)
print("month range:", final_df["month"].iloc[0], "to", final_df["month"].iloc[-1])
print("columns:")
for col in final_df.columns:
    print("-", col)

display(final_df.tail(12))


final file: E:\财遇见你\大四\申万实习\动态因子模型\动态因子模型\data\processed\processed\宏观变量&GDP raw data.csv
shape: (556, 15)
month range: 1980-01 to 2026-04
columns:
- month
- 中国:GDP:现价:当季值
- 中国:GDP:不变价:当季值
- 万得一致预测:中国:GDP:不变价:当季同比
- 中国:制造业PMI
- 中国:非制造业PMI:服务业
- 中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分)
- 中国:全社会用电量:当月同比
- 中国:社会消费品零售总额:当月同比(1-2月合并)
- 中国:固定资产投资完成额:当月同比
- 中国:CPI:当月同比
- 中国:PPI:当月同比
- 中国:城镇调查失业率
- 中国:GDP:现价:当季值同比
- 中国:GDP:不变价:当季值同比


,month,中国:GDP:现价:当季值,中国:GDP:不变价:当季值,万得一致预测:中国:GDP:不变价:当季同比,中国:制造业PMI,中国:非制造业PMI:服务业,中国:工业增加值:规模以上工业企业:当月同比(1-2月拆分),中国:全社会用电量:当月同比,中国:社会消费品零售总额:当月同比(1-2月合并),中国:固定资产投资完成额:当月同比,中国:CPI:当月同比,中国:PPI:当月同比,中国:城镇调查失业率,中国:GDP:现价:当季值同比,中国:GDP:不变价:当季值同比
544,2025-05,NaN,NaN,NaN,49.5,50.2,5.8,4.431500,6.4,0.036071,-0.1,-3.3,5.0,NaN,NaN
545,2025-06,341395.3,323458.2,5.16,49.7,50.1,6.8,5.421237,4.8,0.271315,0.1,-3.6,5.0,0.038985,0.052345
546,2025-07,NaN,NaN,NaN,49.3,50.0,5.7,8.582166,3.7,-0.310360,0.0,-3.6,5.2,NaN,NaN
547,2025-08,NaN,NaN,NaN,49.4,50.5,5.2,4.990763,3.4,-0.102748,-0.4,-2.9,5.3,NaN,NaN
548,2025-09,354106.2,341216.5,4.76,49.8,50.1,6.5,4.536700,3.0,0.087375,-0.3,-2.3,5.2,0.037087,0.048351
549,2025-10,NaN,NaN,NaN,49.0,50.2,4.9,10.356738,2.9,-0.246285,0.2,-2.1,5.1,NaN,NaN
550,2025-11,NaN,NaN,NaN,49.2,49.5,4.8,6.171445,1.3,-0.206197,0.7,-2.2,5.1,NaN,NaN
551,2025-12,387911.3,374444.5,4.48,50.1,49.7,5.2,2.770855,0.9,-0.034399,0.8,-1.9,5.1,0.038550,0.044576
552,2026-01,NaN,NaN,NaN,49.3,49.5,6.3,NaN,NaN,NaN,0.2,-1.4,5.2,NaN,NaN
553,2026-02,NaN,NaN,NaN,49.0,49.7,6.3,NaN,2.8,0.086247,1.3,-0.9,5.3,NaN,NaN
